1# Step 2.1  — Connect Python to MySQL

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import numpy as np

engine = create_engine(
    "mysql+pymysql://",
    connect_args={
        "host"   : "localhost",
        "user"   : "root",
        "password": "****",  # paste your real password here
        "database": "logistics_db"
    }
)

# Test connection
with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


In [3]:
# Load all tables into DataFrames:

with engine.connect() as conn:
    df_shipments = pd.read_sql("SELECT * FROM shipments",      conn)
    df_customers = pd.read_sql("SELECT * FROM customers",      conn)
    df_carriers  = pd.read_sql("SELECT * FROM carriers",       conn)
    df_products  = pd.read_sql("SELECT * FROM products",       conn)
    df_items     = pd.read_sql("SELECT * FROM shipment_items", conn)

print("Rows loaded:")
print(f"  Shipments      : {len(df_shipments)}")
print(f"  Customers      : {len(df_customers)}")
print(f"  Carriers       : {len(df_carriers)}")
print(f"  Products       : {len(df_products)}")
print(f"  Shipment Items : {len(df_items)}")

Rows loaded:
  Shipments      : 20
  Customers      : 8
  Carriers       : 4
  Products       : 8
  Shipment Items : 20


In [4]:
#  Load full joined dataset (main working table):

query = """
SELECT
    s.shipment_id,
    c.customer_name,
    c.city         AS customer_city,
    w.warehouse_name,
    ca.carrier_name,
    s.shipment_date,
    s.expected_delivery,
    s.actual_delivery,
    s.delivery_status,
    s.destination_city,
    s.freight_cost,
    p.product_name,
    p.category,
    si.quantity,
    si.total_price
FROM shipments s
JOIN customers      c  ON s.customer_id  = c.customer_id
JOIN warehouses     w  ON s.warehouse_id = w.warehouse_id
JOIN carriers       ca ON s.carrier_id   = ca.carrier_id
JOIN shipment_items si ON s.shipment_id  = si.shipment_id
JOIN products       p  ON si.product_id  = p.product_id
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print(f"\nTotal rows in joined dataset: {len(df)}")
print(df.head())


Total rows in joined dataset: 20
   shipment_id            customer_name customer_city  \
0            8         Deepa Cold Chain        Mumbai   
1            4            Priya Exports     Bengaluru   
2            1  Arjun Logistics Pvt Ltd       Chennai   
3           11      Karthik Enterprises    Coimbatore   
4           15        Suresh Auto Parts          Pune   

              warehouse_name       carrier_name shipment_date  \
0    Mumbai Western Facility  Blue Dart Express    2024-02-14   
1        Bengaluru South Hub  Blue Dart Express    2024-01-20   
2  Chennai Central Warehouse  Blue Dart Express    2024-01-05   
3  Chennai Central Warehouse  Blue Dart Express    2024-03-05   
4  Chennai Central Warehouse  Blue Dart Express    2024-04-01   

  expected_delivery actual_delivery delivery_status destination_city  \
0        2024-02-17      2024-02-20         Delayed           Mumbai   
1        2024-01-23      2024-01-25         Delayed        Bengaluru   
2        2024-01

## Step 2.2 — Clean the data

In [5]:
# Check data types and nulls:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nNull values:")
print(df.isnull().sum())

Shape: (20, 15)

Data types:
shipment_id            int64
customer_name            str
customer_city            str
warehouse_name           str
carrier_name             str
shipment_date         object
expected_delivery     object
actual_delivery       object
delivery_status          str
destination_city         str
freight_cost         float64
product_name             str
category                 str
quantity               int64
total_price          float64
dtype: object

Null values:
shipment_id          0
customer_name        0
customer_city        0
warehouse_name       0
carrier_name         0
shipment_date        0
expected_delivery    0
actual_delivery      3
delivery_status      0
destination_city     0
freight_cost         0
product_name         0
category             0
quantity             0
total_price          0
dtype: int64


In [6]:
#  Fix date columns:
df['shipment_date']     = pd.to_datetime(df['shipment_date'])
df['expected_delivery'] = pd.to_datetime(df['expected_delivery'])
df['actual_delivery']   = pd.to_datetime(df['actual_delivery'])

print("Date columns fixed.")
print(df[['shipment_date','expected_delivery','actual_delivery']].dtypes)

Date columns fixed.
shipment_date        datetime64[s]
expected_delivery    datetime64[s]
actual_delivery      datetime64[s]
dtype: object


In [7]:
#Add useful calculated columns:

# Delay in days (actual - expected). Negative = early, Positive = delayed
df['delay_days'] = (df['actual_delivery'] - df['expected_delivery']).dt.days

# Month label for trend analysis
df['shipment_month'] = df['shipment_date'].dt.to_period('M').astype(str)

# Flag: was this shipment delayed?
df['is_delayed'] = df['delivery_status'] == 'Delayed'

print("New columns added: delay_days, shipment_month, is_delayed")
df[['shipment_id','delivery_status','delay_days','shipment_month','is_delayed']].head(8)

New columns added: delay_days, shipment_month, is_delayed


,shipment_id,delivery_status,delay_days,shipment_month,is_delayed
0,8,Delayed,3.0,2024-02,True
1,4,Delayed,2.0,2024-01,True
2,1,Delivered,0.0,2024-01,False
3,11,Delayed,2.0,2024-03,True
4,15,Delivered,0.0,2024-04,False
5,19,In Transit,NaN,2024-04,False
6,10,Delivered,0.0,2024-03,False
7,17,Delivered,0.0,2024-04,False


## Step 2.3 — Analyse the data (KPIs & GroupBy)

In [8]:
# Overall KPIs:

total_shipments  = df['shipment_id'].nunique()
total_revenue    = df['total_price'].sum()
total_freight    = df['freight_cost'].sum()
on_time_rate     = (df[df['delivery_status'] == 'Delivered']['shipment_id'].nunique() /
                    total_shipments * 100)
delay_rate       = (df[df['delivery_status'] == 'Delayed']['shipment_id'].nunique() /
                    total_shipments * 100)

print("======= KEY PERFORMANCE INDICATORS =======")
print(f"Total Shipments   : {total_shipments}")
print(f"Total Revenue     : ₹{total_revenue:,.2f}")
print(f"Total Freight Cost: ₹{total_freight:,.2f}")
print(f"On-Time Rate      : {on_time_rate:.1f}%")
print(f"Delay Rate        : {delay_rate:.1f}%")

======= KEY PERFORMANCE INDICATORS =======
Total Shipments   : 20
Total Revenue     : ₹186,000.00
Total Freight Cost: ₹27,350.00
On-Time Rate      : 55.0%
Delay Rate        : 30.0%


In [9]:
# Revenue by Category:
category_revenue = (df.groupby('category')['total_price']
                      .sum()
                      .reset_index()
                      .rename(columns={'total_price':'revenue'})
                      .sort_values('revenue', ascending=False))

print("Revenue by Category:")
print(category_revenue)

Revenue by Category:
       category  revenue
3   Electronics  48000.0
0    Automobile  31500.0
1     Chemicals  25600.0
6  Raw Material  22000.0
2  Construction  18000.0
5    Healthcare  18000.0
4          FMCG  13300.0
7       Textile   9600.0


In [10]:
# Carrier performance (delay rate per carrier):

carrier_perf = df.groupby('carrier_name').agg(
    total_shipments = ('shipment_id', 'nunique'),
    delayed         = ('is_delayed',  'sum'),
).reset_index()

carrier_perf['delay_rate_%'] = (carrier_perf['delayed'] /
                                 carrier_perf['total_shipments'] * 100).round(1)

carrier_perf = carrier_perf.sort_values('delay_rate_%', ascending=False)
print("Carrier Performance:")
print(carrier_perf)

Carrier Performance:
        carrier_name  total_shipments  delayed  delay_rate_%
0  Blue Dart Express                6        3          50.0
1       DTDC Courier                5        2          40.0
2  Delhivery Pvt Ltd                5        1          20.0
3       Ecom Express                4        0           0.0


In [11]:
# Monthly shipment volume trend:
monthly = (df.groupby('shipment_month')['shipment_id']
             .nunique()
             .reset_index()
             .rename(columns={'shipment_id':'shipments'}))

print("Monthly Shipment Volume:")
print(monthly)

Monthly Shipment Volume:
  shipment_month  shipments
0        2024-01          4
1        2024-02          5
2        2024-03          5
3        2024-04          6


In [ ]:
# Top destination cities by freight cost:

city_freight = (df.groupby('destination_city')['freight_cost']
                  .sum()
                  .reset_index()
                  .rename(columns={'freight_cost':'total_freight'})
                  .sort_values('total_freight', ascending=False))

print("Freight Cost by City:")
print(city_freight)

Freight Cost by City:
  destination_city  total_freight
3        Hyderabad         4950.0
5           Mumbai         4300.0
1          Chennai         3750.0
6             Pune         3700.0
2       Coimbatore         3300.0
0        Bengaluru         2900.0
4          Madurai         2670.0
7         Tiruppur         1780.0


In [13]:
conn.close()
print("MySQL connection closed.")

MySQL connection closed.
